# Part 1: Data Cleaning and EDA


## Background: the PECARN TBI study

This data comes from a prospective study PECARN ran across 25 emergency departments (Kuppermann et al., *Lancet* 2009). Kids under 18 with blunt head trauma were enrolled, and physicians filled out a standardized form before they knew the CT results. About 4% of patients also got a second, independent exam so the study could check how consistent different physicians' ratings were, and anyone sent home without a CT got a follow-up call 7-90 days later so a missed injury wouldn't just vanish from the data.

The rule Kuppermann's team eventually published was derived only on patients with a GCS of 14-15 (relatively normal), split into an under-2 rule and a 2-and-older rule, since younger kids can't really describe symptoms like headache or amnesia. However, I kept every enrolled patient no matter their GCS. That difference comes up more than once below, both in why some symptom columns are missing so much and in a gap I find later between our ciTBI rate and the paper's.

The outcome column, `PosIntFinal`, is the paper's ciTBI definition: death from TBI, neurosurgery, intubation for more than 24h for head trauma, or 2+ nights hospitalized with a TBI finding on CT.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from clean import clean_data

In [2]:
# Relative to code/ (the notebook's working directory); data/ is not tracked by git
data_path = Path('..') / 'data' / 'TBI PUD 10-08-2013.csv'
raw = pd.read_csv(data_path)
print('raw shape:', raw.shape)
raw.head()

raw shape: (43399, 125)


,PatNum,EmplType,Certification,InjuryMech,High_impact_InjSev,Amnesia_verb,LOCSeparate,LocLen,Seiz,SeizOccur,...,Finding20,Finding21,Finding22,Finding23,DeathTBI,HospHead,HospHeadPosCT,Intub24Head,Neurosurgery,PosIntFinal
0,1,3.0,3,11.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
1,2,5.0,3,8.0,2.0,0.0,0.0,92.0,0.0,92.0,...,0,0,0,0,0.0,0.0,0,0.0,0.0,0.0
2,3,5.0,3,5.0,2.0,NaN,NaN,92.0,NaN,92.0,...,0,0,0,0,0.0,1.0,0,0.0,0.0,0.0
3,4,5.0,3,6.0,1.0,91.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
4,5,3.0,3,12.0,2.0,91.0,0.0,92.0,0.0,92.0,...,0,0,0,0,0.0,0.0,0,0.0,0.0,0.0


## 1. Data quality checks

Ran a few diagnostic checks on the raw data to see its quality: structural integrity, missingness, sentinel/special-value codes.

In [3]:
print('shape:', raw.shape)
print('duplicate PatNum rows:', raw['PatNum'].duplicated().sum())
print('any null PatNum:', raw['PatNum'].isna().sum())
print()
print('dtype counts:')
print(raw.dtypes.value_counts())

shape: (43399, 125)
duplicate PatNum rows: 0
any null PatNum: 0

dtype counts:
int64      77
float64    48
Name: count, dtype: int64


In [4]:
raw_na_rate = raw.isna().mean().sort_values(ascending=False)
print('columns with any raw (true) NaN:', (raw_na_rate > 0).sum(), 'of', raw.shape[1])
raw_na_rate.head(15)

columns with any raw (true) NaN: 48 of 125


Dizzy           0.368027
Ethnicity       0.367889
ActNorm         0.076845
Race            0.073919
LocLen          0.058895
Observed        0.054748
Amnesia_verb    0.052904
LOCSeparate     0.043595
Drugs           0.041890
HAStart         0.030692
GCSMotor        0.030185
GCSVerbal       0.029909
GCSEye          0.029678
HASeverity      0.024056
VomitLast       0.022858
dtype: float64

In [5]:
sentinel_codes = [90, 91, 92]
for code in sentinel_codes:
    counts = (raw == code).sum()
    counts = counts[counts > 0].sort_values(ascending=False)
    print(f'--- sentinel {code}: present in {len(counts)} columns ---')
    print(counts.head(10))
    print()

--- sentinel 90: present in 6 columns ---
InjuryMech       3465
Race             1339
Certification     718
EDDisposition     248
AgeInMonth        149
PatNum              1
dtype: int64

--- sentinel 91: present in 4 columns ---
Amnesia_verb    14805
HA_verb         14060
AgeInMonth        139
PatNum              1
dtype: int64



--- sentinel 92: present in 82 columns ---
SFxPalpDepress    43175
SFxBasOto         43003
SFxBasRhi         43003
SFxBasRet         43003
SFxBasPer         43003
SFxBasHem         43003
SeizOccur         42796
SeizLen           42796
CTSedAgitate      42745
CTSedAge          42745
dtype: int64



In [6]:
exclude_cols = {'PatNum', 'AgeInMonth'}
unexpected = {}
for col in raw.columns:
    if col in exclude_cols:
        continue
    vals = pd.to_numeric(raw[col], errors='coerce').dropna().unique()
    stray = [v for v in vals if v >= 90 and v not in (90, 91, 92)]
    if stray:
        unexpected[col] = sorted(stray)

print('columns with sentinel-like values other than 90/91/92:', len(unexpected))
unexpected

columns with sentinel-like values other than 90/91/92: 0


{}

In [7]:
print('PosIntFinal value counts (incl. NaN):')
print(raw['PosIntFinal'].value_counts(dropna=False))
print()
print('PosIntFinal positive rate:', round(raw['PosIntFinal'].mean(), 4))
print()
print('CTDone value counts (incl. NaN):')
print(raw['CTDone'].value_counts(dropna=False))

PosIntFinal value counts (incl. NaN):
PosIntFinal
0.0    42616
1.0      763
NaN       20
Name: count, dtype: int64

PosIntFinal positive rate: 0.0176

CTDone value counts (incl. NaN):
CTDone
0    27500
1    15899
Name: count, dtype: int64


In [8]:
print('--- Comparability: are derived variables internally consistent? ---')
age_mismatch = (raw['AgeinYears'] != (raw['AgeInMonth'] // 12)).sum()
print(f'AgeinYears != floor(AgeInMonth / 12): {age_mismatch} of {len(raw)} rows')

gcs_group_expected = np.where(raw['GCSTotal'] >= 14, 2, 1)
gcs_mismatch = (raw['GCSGroup'] != gcs_group_expected).sum()
print(f'GCSGroup inconsistent with (GCSTotal >= 14): {gcs_mismatch} of {len(raw)} rows')
print()

print('--- Meaning: is missingness explained by the < 2 years exclusion? ---')
under2 = raw['AgeinYears'] < 2
for col in ['Dizzy', 'Amnesia_verb', 'HA_verb']:
    print(f'{col}: NaN rate under 2y = {raw.loc[under2, col].isna().mean():.1%}, '
          f'2y and older = {raw.loc[~under2, col].isna().mean():.1%}')

--- Comparability: are derived variables internally consistent? ---
AgeinYears != floor(AgeInMonth / 12): 0 of 43399 rows
GCSGroup inconsistent with (GCSTotal >= 14): 0 of 43399 rows

--- Meaning: is missingness explained by the < 2 years exclusion? ---
Dizzy: NaN rate under 2y = 87.5%, 2y and older = 19.8%
Amnesia_verb: NaN rate under 2y = 3.4%, 2y and older = 5.9%
HA_verb: NaN rate under 2y = 0.8%, 2y and older = 1.7%


Both checks come back clean across all 43,399 rows: `AgeinYears` is just `floor(AgeInMonth / 12)`, and `GCSGroup` is just `GCSTotal >= 14`. So they're not two measurements that could disagree - it's the same value stored twice.

The age-based missingness isn't random either. The paper says amnesia, headache, and dizziness weren't assessed in kids under 2, which makes sense since they can't really report those. The codebook backs this up: `Amnesia_verb` and `HA_verb` have an actual `91 = pre-verbal/non-verbal` code, so their missing rate stays low even under age 2 (3.4% and 0.8%). `Dizzy` has no such option in its format - just Yes/No - so for kids under 2 it's simply left blank, which is why it's missing 87.5% of the time there versus 19.8% for 2 and up. That explains why `Dizzy` (along with `Ethnicity`) stood out earlier as one of the biggest missingness columns: at least for `Dizzy`, it's not noise, it's the protocol.

### Data check summary

- Structure is clean at the row level. No duplicate or missing `PatNum`;
  every column is already numeric (int64/float64), so no string-parsing
  cleanup is needed.
- True (CSV-native) NaNs are limited to 48 of 125 columns and are mostly
  low-rate, with `Dizzy` and `Ethnicity` as outliers (~37% missing) and
  everything else under ~8%. `Dizzy` is missing a lot more under age 2
  (87.5% vs. 19.8%), which lines up with the paper's note that dizziness
  isn't assessed pre-verbally - so it's a real "not asked" rather than a
  random gap. These are genuine missing/not-assessed responses, distinct
  from the sentinel codes below, and are left as NaN rather than imputed
  at the cleaning stage.
- Sentinel 92 ("Not applicable") is by far the most common special code
- Sentinels 90 ("Other") and 91 ("Pre-verbal/Non-verbal") are much rarer
  and concentrated in specific columns.
- No stray out-of-codebook values
- `AgeinYears`/`AgeInMonth` and `GCSGroup`/`GCSTotal` are just the same
  value stored two ways (0 mismatches across all rows), not two
  measurements that could actually disagree.
- The modeling target is heavily imbalanced: `PosIntFinal` is 1.76%
  positive, with 20 missing labels (dropped via `drop_missing_target=True`
  when needed).

## 2. Data cleaning

Based on the above data quality checks, the following cleaning steps are applied to the raw data:
The raw CSV uses a large number of sentinel values such as 90, 91, and 92 to encode things like 'other', 'not applicable', and 'missing'. These are not ordinary numeric values and should be treated/cleaned systematically. Based on TBI PUD Documentation 10-08-2013.xlsx, I asked generative AI to list all the variables that might contain value 90/91/92 using prompt: "List the variables in the dataset that might contain the sentinel values 90, 91,92" and list them out in clean.py for cleaning process.

I had to make the judgmenet call here: how to treat these sentinel values. I decided to only turn 92 into NaN, and leave 90 and 91 as-is. The reasoning is that 90 and 91 are valid responses that indicate a specific condition (e.g., "Other" or "Pre-verbal/Non-verbal"), while 92 indicates that the question was not applicable to the respondent, which is more akin to missing data.

In [9]:
df = clean_data(raw)
print('clean shape:', df.shape)
print('missing values per column (top 10):')
print(df.isna().mean().sort_values(ascending=False).head(10))
print('target prevalence:')
print(df['PosIntFinal'].mean())
df[['PatNum', 'AgeinYears', 'HA_verb', 'Vomit', 'GCSGroup', 'PosIntFinal']].head()

clean shape: (43399, 125)
missing values per column (top 10):
SFxPalpDepress    0.996106
SFxBasHem         0.990875
SFxBasOto         0.990875
SFxBasPer         0.990875
SFxBasRet         0.990875
SFxBasRhi         0.990875
SeizLen           0.988802
SeizOccur         0.987742
CTSedOth          0.984931
CTSedRqst         0.984931
dtype: float64
target prevalence:
0.01758915604324673


,PatNum,AgeinYears,HA_verb,Vomit,GCSGroup,PosIntFinal
0,1,16,1.0,0.0,2,0.0
1,2,5,0.0,1.0,2,0.0
2,3,14,NaN,NaN,1,0.0
3,4,1,91.0,0.0,2,0.0
4,5,1,91.0,1.0,1,0.0


## 3. Exploratory data analysis

The main clinical question: how likely is a patient to have a clinically significant TBI, and how do symptoms like headache, vomiting, or altered mental status change that risk? A useful screening rule has to be sensitive, easy to use, and interpretable. Headache, vomiting, altered mental status, GCS, and injury severity aren't a random pick either. They're the same categories of variable Kuppermann et al. used to build their published rule, so they seem like a reasonable place to start.

In [10]:
age_summary = (
    df.groupby('AgeinYears', dropna=True)['PosIntFinal']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'TBI_positive_rate', 'count': 'n'})
    .reset_index()
)

age_summary = age_summary[age_summary['n'] >= 20].sort_values('AgeinYears')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(age_summary['AgeinYears'], age_summary['TBI_positive_rate'],
        marker='o', markersize=6, linewidth=2, color='#2a78d6')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Positive final ciTBI rate')
ax.set_title('Clinically-important TBI (ciTBI) rate by age')
ax.set_xticks(range(0, 18, 2))
ax.grid(alpha=0.25, color='#c3c2b7')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
ax.text(0.98, 0.02, f'n = {age_summary["n"].sum():,} patients (ages with n ≥ 20)',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8, color='#52514e')
plt.tight_layout()
plt.show()

<Figure size 3000x1500 with 1 Axes>

### Finding 1: rates rise with age and are concentrated in older children

The rate isn't flat across ages. From the graph, it climbs for older kids, which fits with older patients being more likely to get hurt by higher-impact mechanisms. That makes age, and how it interacts with mechanism and symptoms, worth building into a risk model. It's probably also part of why Kuppermann et al. split their rule at age 2 instead of using one rule for everyone: both risk and how well a kid can report symptoms change on either side of that line.

In [11]:
symptom_cols = ['HA_verb', 'Vomit', 'AMS', 'Intubated', 'Sedated']
symptom_labels = {
    'HA_verb': 'Headache',
    'Vomit': 'Vomiting',
    'AMS': 'Altered mental status',
    'Intubated': 'Intubated',
    'Sedated': 'Sedated',
}

rows = []
for col in symptom_cols:
    # Restrict to the two substantive Yes/No codes; HA_verb also carries a
    # 91 ("pre-verbal") code that isn't a "No" and shouldn't be pooled with it.
    sub = df[df[col].isin([0, 1])][[col, 'PosIntFinal']].dropna()
    grouped = sub.groupby(col)['PosIntFinal'].agg(['mean', 'count'])
    for present, (rate, n) in grouped.iterrows():
        rows.append({'symptom': symptom_labels[col], 'present': 'Yes' if present == 1 else 'No', 'rate': rate, 'n': n})
plot_df = pd.DataFrame(rows)

order = [symptom_labels[c] for c in symptom_cols]
pivot_rate = plot_df.pivot(index='symptom', columns='present', values='rate').loc[order]
pivot_n = plot_df.pivot(index='symptom', columns='present', values='n').loc[order]

fig, ax = plt.subplots(figsize=(9, 5))
pivot_rate[['No', 'Yes']].plot(kind='bar', ax=ax, color=['#2a78d6', '#eb6834'], width=0.75)
for container, col in zip(ax.containers, ['No', 'Yes']):
    labels = [f'n={int(n):,}' for n in pivot_n[col]]
    ax.bar_label(container, labels=labels, fontsize=7, color='#52514e', padding=2)
ax.set_title('Observed ciTBI rate by symptom presence')
ax.set_ylabel('Positive ciTBI rate')
ax.set_xlabel('')
ax.legend(title='Symptom present', frameon=False)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

<Figure size 2700x1500 with 1 Axes>

### Finding 2: symptoms such as headache and vomiting are strongly linked to an elevated positive-TBI rate

This lines up with how ERs actually work: headache, vomiting, and altered mental status are classic red flags for ordering a CT, and three of them are literally in Kuppermann's rule. The association isn't necessarily causal, but it does suggest a handful of symptoms carry most of the signal here.

`Intubated` and `Sedated` show the biggest jumps of all (74% and 57% ciTBI when present, versus ~1% otherwise), but I'd read these two differently than the other three. They don't capture a symptom the patient had before the CT decision. They capture whether the exam happened after the patient was already intubated or sedated, which itself means the team had already decided this was serious. They're a good example of a variable that's strongly associated with the outcome but not something you could use to screen a patient at triage. That's probably why Kuppermann's rule skips them too.

### A closer look: headache severity

Kuppermann et al.'s rule for kids 2 and older actually uses "no severe headache" as a rule variable, so this isn't a throwaway column. It's also our most sentinel-heavy variable - `HASeverity` is coded 92 ("not applicable") for about 70% of patients who never reported a headache, which makes it a good test case for whether the 92-cleaning decision actually matters anywhere.

In [12]:
ha_severity_num = pd.to_numeric(df['HASeverity'], errors='coerce')
ha_severe = (ha_severity_num >= 2).where(ha_severity_num.notna())
n_recorded = ha_severe.notna().sum()
rate_severe = ha_severe.dropna().astype(float).mean()
print(f'Patients with a recorded headache severity: {n_recorded:,}')
print(f'Share with moderate-or-severe headache: {rate_severe:.1%}')

ha_severe_df = pd.DataFrame({'HA_severe': ha_severe, 'PosIntFinal': df['PosIntFinal']}).dropna()
print()
print('ciTBI rate by headache severity:')
print(ha_severe_df.groupby('HA_severe')['PosIntFinal'].agg(['mean', 'count']))

Patients with a recorded headache severity: 11,762
Share with moderate-or-severe headache: 55.2%

ciTBI rate by headache severity:
               mean  count
HA_severe                 
False      0.004748   5265
True       0.016949   6490


Of the 11,762 patients who did report a headache, 55.2% call it moderate or severe, and their ciTBI rate (1.7%) is higher than for a mild headache (0.5%). That's a decent sign "severe headache" is picking up something real.

It's also the one place in this notebook where the sentinel-cleaning choice actually changes the answer, which we check next.

## 4. Reality check

Our cleaned data gives a ciTBI rate of 1.76% and a CT rate of 36.6%. Kuppermann et al. give us something to check that against: across their derivation and validation cohorts, ciTBI showed up in 376 of 42,412 kids (0.9%), with about 35% getting a CT. Both cohorts restricted to GCS 14-15, since that's who the rule is for. Our data keeps everyone regardless of GCS, so the two numbers aren't quite measuring the same population.

In [13]:
print('Positive final TBI rate overall:', round(df['PosIntFinal'].mean(), 4))
print('Patients with CT done:', round(df['CTDone'].mean(), 4))
print('Positive rate among those with CT done:', round(df.loc[df['CTDone'] == 1, 'PosIntFinal'].mean(), 4))
print('Positive rate among those without CT done:', round(df.loc[df['CTDone'] == 0, 'PosIntFinal'].mean(), 4))
print()
print('--- Comparison to Kuppermann et al. (2009) ---')
print('Paper (GCS 14-15 only): 376/42,412 = 0.9% ciTBI')
print('Our full PUD sample (all GCS):', round(df['PosIntFinal'].mean(), 4))
gcs1415 = df[df['GCSGroup'] == 2]
print(f'Our sample restricted to GCS 14-15 (n={len(gcs1415):,}):', round(gcs1415['PosIntFinal'].mean(), 4))

Positive final TBI rate overall: 0.0176
Patients with CT done: 0.3663
Positive rate among those with CT done: 0.0468
Positive rate among those without CT done: 0.0007

--- Comparison to Kuppermann et al. (2009) ---
Paper (GCS 14-15 only): 376/42,412 = 0.9% ciTBI
Our full PUD sample (all GCS): 0.0176
Our sample restricted to GCS 14-15 (n=42,430): 0.0089


Restricting to GCS 14-15 brings our rate down to 0.89% - almost exactly the paper's 0.9%. That's a good sign: the roughly 2x gap between our headline number and the paper's isn't a cleaning mistake, it's just that our data includes the sicker, lower-GCS kids the paper's cohorts excluded on purpose. Those kids are more likely to have a ciTBI, which is exactly why you'd derive a "can we skip the CT" rule on the lower-risk group in the first place. This also matches the dataset's own documentation, which says the public-use file keeps everyone "regardless of GCS score."

## 5. Stability check

Two judgment calls went into the sentinel cleaning, and they're not equally debatable. Recoding 92 ("not applicable") to NaN is the obvious one - "not applicable" is about as clear a synonym for missing as a codebook gets. The more interesting question is whether 90 ("other") and 91 ("pre-verbal/non-verbal") should also have been recoded to NaN instead of being kept as real category values. I will check the "obvious" 92 call too, just to confirm it's actually as safe as it looks.

### 90 and 91: the real judgment call

`InjuryMech`'s 90 ("other") and `HA_verb`'s/`Amnesia_verb`'s 91 ("pre-verbal/non-verbal") are left as real category values instead of being recoded to NaN, on the reasoning that they're substantive answers rather than missing data.

In [14]:
# HA_verb carries a 91 ("pre-verbal") code sitting on the same 0/1 scale
# as a real answer, so an unfiltered mean is exactly the kind of naive
# computation that broke on HASeverity above.
naive_kept = df['HA_verb'].astype(float).mean()

ha_verb_recoded = df['HA_verb'].astype(float).replace(91, np.nan)
naive_recoded = ha_verb_recoded.mean()

# Finding 2 never took a naive mean, though - it filtered to isin([0, 1])
# first, so the two cleaning choices should agree there.
proper_kept = df[df['HA_verb'].isin([0, 1])]['HA_verb'].astype(float).mean()
proper_recoded = ha_verb_recoded.dropna().mean()

print(f'naive mean, 91 kept as a literal code: {naive_kept:.2f}')
print(f'naive mean, 91 -> NaN:                {naive_recoded:.3f}')
print()
print(f'Finding-2-style rate, 91 kept:   {proper_kept:.4f}')
print(f'Finding-2-style rate, 91 -> NaN: {proper_recoded:.4f}')

naive mean, 91 kept as a literal code: 30.23
naive mean, 91 -> NaN:                0.446

Finding-2-style rate, 91 kept:   0.4464
Finding-2-style rate, 91 -> NaN: 0.4464


A naive `.mean()` on `HA_verb` breaks the same way `HASeverity` did: 91 sits on the same column as the real 0/1 answers, so an unfiltered average comes out around 30, which is obviously wrong. The difference here is that once 91 is recoded to NaN, pandas' `.mean()` skips it automatically and gets the right answer without us having to remember to filter anything. Once filter properly `isin([0, 1])`, which Finding 2 already does, the two cleaning choices agree exactly (0.4464 either way).

### And 92

Just to make sure the easy call is actually right, here's the same kind of check applied to recoding 92 to NaN. None of the columns behind Finding 1 or Finding 2 are ever coded 92, so we test it on `HASeverity`.

In [15]:
def ha_severe_rate(not_applicable_value):
    d = clean_data(raw, not_applicable_value=not_applicable_value, categorize=False)
    sev = pd.to_numeric(d['HASeverity'], errors='coerce')
    flag = (sev >= 2).where(sev.notna())
    return flag.notna().sum(), flag.dropna().astype(float).mean()

# Before: 92 recoded to NaN before thresholding (our actual cleaning choice).
n_before, rate_before = ha_severe_rate(np.nan)
# After: 92 left as a literal numeric code, so it satisfies ">= 2" for
# every patient who never even reported a headache.
n_after, rate_after = ha_severe_rate(92)

print(f'before (92 -> NaN):  n={n_before:,}, moderate/severe rate={rate_before:.1%}')
print(f'after (92 kept):     n={n_after:,}, moderate/severe rate={rate_after:.1%}')

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(['Before\n(92 → NaN)', 'After\n(92 kept as a code)'],
              [rate_before, rate_after], color=['#2a78d6', '#e34948'], width=0.55)
ax.bar_label(bars, labels=[f'{rate_before:.1%}\n(n={n_before:,})', f'{rate_after:.1%}\n(n={n_after:,})'], padding=4)
ax.set_ylabel('Share flagged "moderate/severe headache"')
ax.set_title('Stability check: does treating 92 as NaN matter?')
ax.set_ylim(0, 1.0)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

before (92 -> NaN):  n=11,762, moderate/severe rate=55.2%
after (92 kept):     n=42,355, moderate/severe rate=87.6%


<Figure size 1800x1500 with 1 Axes>

This confirms the obvious answer: leaving 92 uncleaned reclassifies most of the "no headache" group as "severe" (87.6% vs. the correct 55.2%), just because 92 happens to be >= 2, and nearly quadruples the apparent sample size (42,355 vs. 11,762). So 92 was the easy call to make, but not a low-stakes one.

## 6. Modeling

Two classification approaches: a logistic regression as a simple, interpretable baseline, and a decision tree as a nonlinear alternative that, like Kuppermann's own rule, can be read off as a small set of yes/no splits. I capped the tree's `max_depth` at 4 on purpose, not to tune for accuracy: a decision rule clinicians can actually remember at the bedside has to stay shallow, which is the same constraint that shaped the real PECARN rule. The goal isn't to maximize accuracy, it's to see how the data drive the risk estimate, and whether either model gets anywhere near the very high sensitivity a screening rule needs.

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

feature_cols = ['AgeinYears', 'InjuryMech', 'HA_verb', 'Vomit', 'AMS', 'GCSGroup', 'Intubated', 'Sedated', 'High_impact_InjSev']
numeric_cols = ['AgeinYears', 'High_impact_InjSev']
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

model_df = df[feature_cols + ['PosIntFinal']].dropna(subset=['PosIntFinal']).copy()
X = model_df[feature_cols]
y = model_df['PosIntFinal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=-1)), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
])

results = []
fitted_models = {}
for weighting, class_weight in [('unweighted', None), ('balanced', 'balanced')]:
    for name, clf in [('logistic_regression', LogisticRegression(max_iter=2000, class_weight=class_weight)),
                       ('decision_tree', DecisionTreeClassifier(max_depth=4, random_state=42, class_weight=class_weight))]:
        pipe = Pipeline([('preprocess', preprocessor), ('model', clf)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        score = pipe.predict_proba(X_test)[:, 1]
        results.append({
            'model': name, 'weighting': weighting,
            'accuracy': accuracy_score(y_test, pred),
            'sensitivity': recall_score(y_test, pred),
            'roc_auc': roc_auc_score(y_test, score),
        })
        fitted_models[(name, weighting)] = pipe

results_df = pd.DataFrame(results).set_index(['model', 'weighting']).round(4)
print(results_df)

# The domain problem wants high sensitivity, so use the balanced fits as
# our two models going forward.
logit = fitted_models[('logistic_regression', 'balanced')]
tree = fitted_models[('decision_tree', 'balanced')]

                                accuracy  sensitivity  roc_auc
model               weighting                                 
logistic_regression unweighted    0.9843       0.2408   0.9145
decision_tree       unweighted    0.9843       0.2408   0.8878
logistic_regression balanced      0.8630       0.7749   0.9132
decision_tree       balanced      0.8542       0.8063   0.8937


### Interpretability

The unweighted fits hit ~98% accuracy but only ~24% sensitivity. Makes sense since ciTBI is only 1.76% of the data. Weighting the loss by class (`class_weight='balanced'`) trades a lot of that accuracy for a lot more sensitivity (roughly 77-81%), which is the right trade for a screening tool, though still well under the 96-100% sensitivity Kuppermann et al. report for their own rule. A handful of general features fit without much tuning just isn't a substitute for a rule that was purpose-built and clinically validated.

Both models are simple enough to read directly: the logistic regression's biggest coefficients and the tree's top splits are below.

In [17]:
from sklearn.tree import export_text

feat_names = logit.named_steps['preprocess'].get_feature_names_out()

coefs = pd.Series(logit.named_steps['model'].coef_[0], index=feat_names)
print('Logistic regression: 10 largest-magnitude coefficients')
print(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(10).round(3))
print()
print('Decision tree: top splits (depth <= 3)')
print(export_text(tree.named_steps['model'], feature_names=list(feat_names), max_depth=3))

Logistic regression: 10 largest-magnitude coefficients
num__High_impact_InjSev    1.573
cat__GCSGroup_2.0         -1.190
cat__AMS_1.0               1.110
cat__InjuryMech_9.0       -1.079
cat__HA_verb_0.0          -1.034
cat__AMS_0.0              -0.956
cat__Vomit_0.0            -0.907
cat__InjuryMech_8.0       -0.895
cat__Sedated_-1.0         -0.817
cat__HA_verb_-1.0          0.817
dtype: float64

Decision tree: top splits (depth <= 3)
|--- cat__AMS_1.0 <= 0.50
|   |--- num__High_impact_InjSev <= 2.50
|   |   |--- cat__Vomit_0.0 <= 0.50
|   |   |   |--- cat__HA_verb_1.0 <= 0.50
|   |   |   |   |--- class: 0.0
|   |   |   |--- cat__HA_verb_1.0 >  0.50
|   |   |   |   |--- class: 0.0
|   |   |--- cat__Vomit_0.0 >  0.50
|   |   |   |--- cat__HA_verb_-1.0 <= 0.50
|   |   |   |   |--- class: 0.0
|   |   |   |--- cat__HA_verb_-1.0 >  0.50
|   |   |   |   |--- class: 1.0
|   |--- num__High_impact_InjSev >  2.50
|   |   |--- cat__HA_verb_0.0 <= 0.50
|   |   |   |--- cat__Vomit_0.0 <= 0.50
|   

Both models land on the same handful of variables, and the directions make sense: high injury severity, altered mental status, and a lower GCS group push risk up, while a normal GCS, no headache, and no vomiting push it down. The tree's first split is on altered mental status, then injury severity, which is roughly how a clinician would triage a head injury too: mental status and mechanism first, symptoms second. Neither model picks `Intubated`/`Sedated` for an early split despite their huge raw association with the outcome in Finding 2, which fits with the earlier point that those two mostly reflect severity that's already obvious by the time GCS and mental status are known.

### Modeling stability check

None of the 9 features above are ever coded 92, so the sentinel choice from Section 5 can't reach these models as they stand. To see if it would matter, add `HASeverity` as a 10th feature and refit both models under the two cleaning choices.

In [18]:
stability_feature_cols = feature_cols + ['HASeverity']
stability_categorical_cols = categorical_cols + ['HASeverity']

stability_preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=-1)), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), stability_categorical_cols),
])

stability_results = []
for cleaning, not_applicable_value in [('cleaned (92 -> NaN)', np.nan), ('perturbed (92 kept)', 92)]:
    d = clean_data(raw, not_applicable_value=not_applicable_value)
    m_df = d[stability_feature_cols + ['PosIntFinal']].dropna(subset=['PosIntFinal']).copy()
    Xs = m_df[stability_feature_cols]
    ys = m_df['PosIntFinal']
    Xs_train, Xs_test, ys_train, ys_test = train_test_split(Xs, ys, test_size=0.25, random_state=42, stratify=ys)
    for name, clf in [('logistic_regression', LogisticRegression(max_iter=2000, class_weight='balanced')),
                       ('decision_tree', DecisionTreeClassifier(max_depth=4, random_state=42, class_weight='balanced'))]:
        pipe = Pipeline([('preprocess', stability_preprocessor), ('model', clf)])
        pipe.fit(Xs_train, ys_train)
        pred = pipe.predict(Xs_test)
        score = pipe.predict_proba(Xs_test)[:, 1]
        stability_results.append({
            'cleaning': cleaning, 'model': name,
            'accuracy': accuracy_score(ys_test, pred),
            'sensitivity': recall_score(ys_test, pred),
            'roc_auc': roc_auc_score(ys_test, score),
        })

stability_results_df = pd.DataFrame(stability_results).set_index(['model', 'cleaning']).round(4)
print(stability_results_df)

                                         accuracy  sensitivity  roc_auc
model               cleaning                                           
logistic_regression cleaned (92 -> NaN)    0.8719       0.7749   0.9148
decision_tree       cleaned (92 -> NaN)    0.8542       0.8063   0.8929
logistic_regression perturbed (92 kept)    0.8716       0.7749   0.9147
decision_tree       perturbed (92 kept)    0.8542       0.8063   0.8928


Accuracy, sensitivity, and AUC are all within about 0.01 whether 92 is cleaned or not, so both models barely moved. That's a different answer than Section 5 got, and it's worth asking why: the EDA used `HASeverity >= 2` as a single numeric threshold, so leaving 92 in as a literal number corrupted the comparison directly. The models one-hot-encode `HASeverity` instead, so 92 just becomes its own category, not a number being compared to anything. The model ends up treating code 92 about the same as the missing-data category on its own, without being told to.

So this cleaning choice matters a lot for hand-written summary statistics, and much less for models that treat the codes as unordered categories. That's a reason to lean on categorical encodings for variables like this by default. This is not a reason to skip the cleaning step, since a different model (or someone using `HASeverity` as a plain number) could run into the same bug I found in Section 5.

I also reran this for the other judgment call from Section 5, recoding 90 ("other", in `InjuryMech`) and 91 ("pre-verbal", in `HA_verb`) to NaN instead of keeping them as categories.

In [19]:
def recode_90_91(d):
    d = d.copy()
    d['InjuryMech'] = d['InjuryMech'].astype(float).replace(90, np.nan)
    d['HA_verb'] = d['HA_verb'].astype(float).replace(91, np.nan)
    return d

results_90_91 = []
for label, d in [('kept as categories', df), ('90/91 -> NaN', recode_90_91(df))]:
    m_df = d[feature_cols + ['PosIntFinal']].dropna(subset=['PosIntFinal']).copy()
    Xr, yr = m_df[feature_cols], m_df['PosIntFinal']
    Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.25, random_state=42, stratify=yr)
    for name, clf in [('logistic_regression', LogisticRegression(max_iter=2000, class_weight='balanced')),
                       ('decision_tree', DecisionTreeClassifier(max_depth=4, random_state=42, class_weight='balanced'))]:
        pipe = Pipeline([('preprocess', preprocessor), ('model', clf)])
        pipe.fit(Xr_train, yr_train)
        pred = pipe.predict(Xr_test)
        score = pipe.predict_proba(Xr_test)[:, 1]
        results_90_91.append({
            'model': name, '90/91 handling': label,
            'accuracy': accuracy_score(yr_test, pred),
            'sensitivity': recall_score(yr_test, pred),
            'roc_auc': roc_auc_score(yr_test, score),
        })

print(pd.DataFrame(results_90_91).set_index(['model', '90/91 handling']).round(4))

                                        accuracy  sensitivity  roc_auc
model               90/91 handling                                    
logistic_regression kept as categories    0.8630       0.7749   0.9132
decision_tree       kept as categories    0.8542       0.8063   0.8937
logistic_regression 90/91 -> NaN          0.8657       0.7749   0.9122
decision_tree       90/91 -> NaN          0.8554       0.8063   0.8931


Same story as `HASeverity`: sensitivity is identical, and accuracy/AUC move by a few thousandths. One-hot encoding treats 90 and 91 as their own categories no matter which way we make this call, so, ike before, it's a real trap for a naive summary statistic and basically irrelevant to these two models.

## 7. Discussion and conclusion

This lab is basically a full pass through the data science life cycle in miniature: the data collection is complicated, the raw values are full of missingness and coding quirks, and none of the exploratory analysis means much without the clinical context behind it. A handful of readily available variables turn out to carry most of the signal for TBI risk, and the cleaning choices, especially around the sentinel codes, matter enough that they're worth documenting carefully rather than glossing over.

Thinking about the three realms (data/reality, algorithms/models, future data/reality): the cleaning and comparability checks in Sections 1-2 are mostly about the first one, whether our 43,399 rows actually reflect what happened in these EDs. Section 4's reality check tests that same boundary against an outside source (the paper) instead of just internal consistency. The modeling in Section 6 lives more in the algorithms/models realm? And the stability checks in Sections 5-6 are really about the third realm: whether what we found here would hold up on new data if we'd made a slightly different preprocessing call.

Given how big this dataset is (43k+ patients), sample size was rarely the actual constraint. The real limits were in what got recorded at all, for instance `Dizzy` for kids under 2, and in how well a numeric code stands in for the clinical idea it's supposed to represent, which is basically what the `HASeverity` sentinel bug was about. There's no one-to-one correspondence between the data and reality here; every variable is someone's summary of a specific ED visit, filtered through a form and a coding scheme. That's exactly why a finding shouldn't be trusted until it survives a reality check and a stability check, not just accepted because the numbers came out of a real dataset.

### 8. Academic Integrity ###
I value UC Berkeley's standards of academic integrity, as well as academic integrity more broadly. I believe honesty is essential to academic research because research depends on others being able to trust how ideas, analyses, and conclusions were produced. All analyses and conclusions in this report are my own. Whenever I received ideas or consultation from other students, outside resources, or AI tools, I have acknowledged their use in the relevant part of the report. I also used AI to help refine the wording of portions of this report, but the underlying analysis, decisions, interpretations, and conclusions are my own.

### 9. References ###
Kuppermann, N., Holmes, J. F., Dayan, P. S., et al. (2009). Identification of children at very low risk of clinically-important brain injuries after head trauma: a prospective cohort study. The Lancet, 374(9696), 1160–1170.